# Bodhi Sync Walkthrough
## Extracting Metadata & Semantic Layer from Athena → Building the Bodhi JSON Payload

---

### What this notebook covers

The **Bodhi Sync** process is the bridge between the YumBurger AWS lakehouse and the
Bodhi AI agent. Its job is to:

1. **Extract schema metadata** — the structure (table names, column names, data types) of
   every silver Iceberg table, read directly from Athena.
2. **Extract the semantic/ontology layer** — business vocabulary mappings that tell Bodhi
   what `'revenue'`, `'best customers'`, or `'gluten-free items'` actually mean in SQL terms.
3. **Detect changes incrementally** — using Iceberg snapshot IDs and content hashing so we
   only re-sync tables that actually changed, keeping costs low.
4. **Assemble a JSON payload** — a structured document ready to be posted to the Bodhi API
   (or, as a placeholder, written to S3).

### Why Athena is the single access layer

All data lives in Apache Iceberg tables on S3. The connector **never** reads S3 directly —
that is enforced by an IAM Deny policy. Athena is the only way in:

```
Application → Athena → Lake Formation (permissions) → S3 Iceberg tables
```

This means schema discovery (`DESCRIBE`), data queries, snapshot history
(`$snapshots` metadata table), and ontology reads all go through Athena.

---

### Prerequisites
- AWS credentials configured (`aws configure` or environment variables)
- YumBurger infrastructure deployed (CDK stacks + Iceberg tables loaded)
- Python packages: `boto3`, `pandas`, `rich`

```bash
pip install boto3 pandas rich
```

---
## 0 — Setup & Configuration

We start by importing libraries and setting the core configuration values.
These values come from `config/settings.yaml` in the real connector —
here we set them explicitly so the notebook is self-contained.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import hashlib
import time
from datetime import datetime, timezone
from uuid import uuid4
from pprint import pprint

import boto3
import pandas as pd

# ── Configuration ─────────────────────────────────────────────────────────────
AWS_REGION         = 'us-east-1'
SILVER_DB          = 'yumburger_silver'
SEMANTIC_DB        = 'yumburger_semantic'
WORKGROUP          = 'yumBurger-wg'
RESULTS_BUCKET     = 'yumburger-athena-results-442042510197'
SYNC_BUCKET        = 'yumburger-athena-results-442042510197'
SYNC_PREFIX        = 'bodhi-sync'
ALLOWED_TABLES     = ['supplier', 'menu_item', 'customer', 'order', 'order_line']

# ── AWS session ───────────────────────────────────────────────────────────────
session   = boto3.Session(region_name=AWS_REGION)
athena    = session.client('athena')
s3        = session.client('s3')

print('✓ AWS session created')
print(f'  Region  : {AWS_REGION}')
print(f'  Silver  : {SILVER_DB}')
print(f'  Semantic: {SEMANTIC_DB}')

---
## 0.1 — Athena Query Helper

Every Athena call follows the same pattern:
1. `start_query_execution` — submit the SQL, get back an execution ID
2. Poll `get_query_execution` until state is `SUCCEEDED` or `FAILED`
3. `get_query_results` (paginated) — fetch rows and convert to list-of-dicts

We wrap this in a helper so every cell below stays readable.

In [ ]:
def run_query(sql: str, database: str = SILVER_DB, timeout: int = 60) -> list[dict]:
    """
    Execute SQL on Athena and return results as a list of dicts.
    Uses the yumBurger workgroup so query costs and permissions are scoped correctly.
    """
    # 1. Submit
    resp = athena.start_query_execution(
        QueryString=sql,
        WorkGroup=WORKGROUP,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={
            'OutputLocation': f's3://{RESULTS_BUCKET}/notebook-scratch/'
        }
    )
    exec_id = resp['QueryExecutionId']
    print(f'  → QueryExecutionId: {exec_id}')

    # 2. Poll
    deadline = time.time() + timeout
    while time.time() < deadline:
        result = athena.get_query_execution(QueryExecutionId=exec_id)
        state  = result['QueryExecution']['Status']['State']
        if state == 'SUCCEEDED':
            break
        if state in ('FAILED', 'CANCELLED'):
            reason = result['QueryExecution']['Status'].get('StateChangeReason', 'unknown')
            raise RuntimeError(f'Athena query {state}: {reason}')
        time.sleep(1)
    else:
        raise TimeoutError(f'Query {exec_id} timed out after {timeout}s')

    # 3. Fetch results (paginated)
    paginator = athena.get_paginator('get_query_results')
    rows, columns, first_page = [], [], True
    for page in paginator.paginate(QueryExecutionId=exec_id):
        result_rows = page['ResultSet']['Rows']
        if first_page:
            columns    = [col['VarCharValue'] for col in result_rows[0]['Data']]
            result_rows = result_rows[1:]          # skip the header row
            first_page = False
        for row in result_rows:
            values = [d.get('VarCharValue') for d in row['Data']]
            rows.append(dict(zip(columns, values)))

    return rows


print('✓ run_query helper defined')

---
## Part A — Extract Schema Metadata

Schema metadata tells Bodhi **what tables exist** and **what columns they have**.
Without this, the LLM cannot generate valid SQL — it would not know that
`customer` has a `lifetime_spend` column, or that `order` joins to `order_line`
via `order_id`.

We use two Athena commands:
- `SHOW TABLES IN <database>` — lists all tables in the catalog
- `DESCRIBE <database>.<table>` — returns column name, data type, and comment for each column

### A.1 — List all silver tables

In [ ]:
print('Listing tables in', SILVER_DB, '...')
rows = run_query(f'SHOW TABLES IN {SILVER_DB}', database=SILVER_DB)

# SHOW TABLES returns a single column — the column name varies by Athena version
tables_in_catalog = [list(r.values())[0] for r in rows]

print(f'\n  Tables found in {SILVER_DB}:')
for t in tables_in_catalog:
    marker = '✓' if t in ALLOWED_TABLES else '—  (not in allowlist)'
    print(f'    {marker}  {t}')

### A.2 — DESCRIBE each table

`DESCRIBE yumburger_silver.customer` returns one row per column with:
- `col_name` — the column name
- `data_type` — the SQL type (e.g. `string`, `double`, `timestamp`)
- `comment` — optional column description

We loop over every table in the allowlist and collect the full schema.

In [ ]:
schema: dict[str, list[dict]] = {}

for table in ALLOWED_TABLES:
    print(f'\nDESCRIBE {table} ...')
    rows = run_query(f'DESCRIBE {SILVER_DB}.{table}', database=SILVER_DB)

    # Filter out Iceberg metadata rows (they appear as empty or '# Partitioning' headers)
    columns = [
        r for r in rows
        if r.get('col_name') and not r['col_name'].startswith('#')
    ]
    schema[table] = columns
    print(f'  → {len(columns)} columns found')

print('\n=== Schema extraction complete ===')

In [ ]:
# Display one table as a DataFrame for readability
table_to_show = 'customer'
df = pd.DataFrame(schema[table_to_show])
print(f'Schema for [{table_to_show}]:')
display(df[['col_name', 'data_type', 'comment']].fillna(''))

### A.3 — Fetch Iceberg Snapshot IDs (incremental tracking)

Apache Iceberg keeps a full history of table changes as **snapshots**.
Each time data is added or changed, Iceberg creates a new snapshot with a unique ID
and a `committed_at` timestamp.

**Why this matters for Bodhi sync:**
If the snapshot ID for `customer` today is the same as it was yesterday, the table
has not changed — no need to re-send that schema to Bodhi. This keeps the sync
cheap and fast.

Iceberg exposes this through a metadata table: `"<db>"."<table>$snapshots"`

In [ ]:
snapshots: dict[str, dict | None] = {}

for table in ALLOWED_TABLES:
    print(f'\nFetching snapshots for {table} ...')
    try:
        rows = run_query(
            f'SELECT * FROM "{SILVER_DB}"."{table}$snapshots" ORDER BY committed_at DESC',
            database=SILVER_DB
        )
        # Latest snapshot is the first row
        snapshots[table] = rows[0] if rows else None
        if rows:
            snap = rows[0]
            print(f'  → snapshot_id   : {snap.get("snapshot_id")}')
            print(f'  → committed_at  : {snap.get("committed_at")}')
            print(f'  → operation     : {snap.get("operation", "n/a")}')
        else:
            print('  → no snapshots found')
    except Exception as e:
        print(f'  ! Could not fetch snapshots: {e}')
        snapshots[table] = None

### A.4 — Build the Schema Payload

Now we assemble the extracted schema and snapshot info into a structured JSON
object — this is the exact format the Bodhi API will eventually consume.

Key fields:
- `payload_type` — tells Bodhi what kind of update this is
- `sync_id` — a UUID so Bodhi can deduplicate retries
- `synced_at` — ISO timestamp for audit purposes
- `tables` — dict of table name → columns + latest snapshot

In [ ]:
sync_id    = str(uuid4())
synced_at  = datetime.now(timezone.utc).isoformat()

schema_payload = {
    'payload_type': 'schema_metadata',
    'sync_id'     : sync_id,
    'synced_at'   : synced_at,
    'source'      : f'athena/{SILVER_DB}',
    'tables'      : {
        table: {
            'columns'         : schema[table],
            'latest_snapshot' : snapshots.get(table)
        }
        for table in ALLOWED_TABLES
    }
}

print('Schema payload assembled.')
print(f'  sync_id   : {schema_payload["sync_id"]}')
print(f'  synced_at : {schema_payload["synced_at"]}')
print(f'  tables    : {list(schema_payload["tables"].keys())}')
print(f'  total cols: {sum(len(v["columns"]) for v in schema_payload["tables"].values())}')

In [ ]:
# Preview the schema payload for one table (truncated to first 3 columns)
preview = json.loads(json.dumps(schema_payload))  # deep copy
for t in preview['tables']:
    preview['tables'][t]['columns'] = preview['tables'][t]['columns'][:3]
    preview['tables'][t]['_note'] = '... (truncated for display)'

print(json.dumps(preview, indent=2, default=str))

---
## Part B — Extract the Semantic / Ontology Layer

The **ontology** is what makes the system intelligent rather than just mechanical.
It is a set of mappings stored in `yumburger_semantic` that translate business
language into SQL language:

| Business term | Maps to |
|---|---|
| `'revenue'` | `order_line.unit_price * order_line.quantity` |
| `'best customers'` | `customer.lifetime_spend DESC` |
| `'gluten-free'` | `menu_item.allergens NOT LIKE '%gluten%'` |

Without these mappings, the LLM would have to guess — and would often guess wrong.
With them, Bedrock receives rich context alongside the question.

Three tables make up the semantic zone:
- **`ontology_class`** — OWL classes, one per silver table (e.g. `Customer`, `Order`)
- **`ontology_property`** — OWL properties, one per important column
- **`ontology_mapping`** — 74 NL-term → table/column mappings

### B.1 — Fetch Ontology Classes

In [ ]:
print('Fetching ontology_class ...')
onto_classes = run_query('SELECT * FROM ontology_class ORDER BY local_name', database=SEMANTIC_DB)

df_classes = pd.DataFrame(onto_classes)
print(f'  → {len(onto_classes)} classes found')
display(df_classes)

### B.2 — Fetch Ontology Properties

In [ ]:
print('Fetching ontology_property ...')
onto_props = run_query('SELECT * FROM ontology_property ORDER BY local_name', database=SEMANTIC_DB)

df_props = pd.DataFrame(onto_props)
print(f'  → {len(onto_props)} properties found')
display(df_props)

### B.3 — Fetch NL→SQL Mappings

These 74 mappings are the most important part of the semantic layer.
Each row links a natural language term to the table and column (or expression)
in the SQL schema that best represents it.

In [ ]:
print('Fetching ontology_mapping ...')
onto_mappings = run_query('SELECT * FROM ontology_mapping ORDER BY nl_term', database=SEMANTIC_DB)

df_map = pd.DataFrame(onto_mappings)
print(f'  → {len(onto_mappings)} mappings found')

# Show a sample of the most useful mappings
sample_terms = ['revenue', 'customer', 'top', 'gluten', 'supplier', 'order']
mask = df_map['nl_term'].str.contains('|'.join(sample_terms), case=False, na=False)
print('\nSample mappings (filtered):')
display(df_map[mask][['nl_term', 'maps_to_table', 'maps_to_column', 'hint']].head(15))

### B.4 — Compute Content Hash (for incremental sync)

Unlike tables, ontology data doesn't have Iceberg snapshots we can compare directly.
Instead we compute an **MD5 hash of the full combined ontology content**.

If the hash today matches the hash stored in the manifest, the ontology has not
changed and we can skip the sync for this run — zero extra Bodhi API calls.

In [ ]:
combined = json.dumps(
    {'classes': onto_classes, 'properties': onto_props, 'mappings': onto_mappings},
    sort_keys=True, default=str
)
content_hash = hashlib.md5(combined.encode()).hexdigest()

print(f'Ontology content hash: {content_hash}')
print('This value is stored in the manifest to detect future changes.')

### B.5 — Build the Ontology Payload

In [ ]:
ontology_payload = {
    'payload_type' : 'ontology',
    'sync_id'      : sync_id,
    'synced_at'    : synced_at,
    'source'       : f'athena/{SEMANTIC_DB}',
    'content_hash' : content_hash,
    'classes'      : onto_classes,
    'properties'   : onto_props,
    'mappings'     : onto_mappings
}

print('Ontology payload assembled.')
print(f'  sync_id      : {ontology_payload["sync_id"]}')
print(f'  content_hash : {ontology_payload["content_hash"]}')
print(f'  classes      : {len(ontology_payload["classes"])}')
print(f'  properties   : {len(ontology_payload["properties"])}')
print(f'  mappings     : {len(ontology_payload["mappings"])}')

---
## Part C — Incremental Sync Logic

Running a full sync on every schedule would be wasteful. The **manifest** solves this.
It is a small JSON file stored in S3 that records:

```json
{
  "snapshots": {
    "customer": "3827491028374910",
    "order":    "9182736450912837",
    ...
  },
  "ontology_hash": "d41d8cd98f00b204e9800998ecf8427e",
  "last_sync": "2026-11-07T14:30:22+00:00"
}
```

**Decision logic:**
- For each table: if `current_snapshot_id != manifest[table]` → re-sync schema
- For ontology: if `current_hash != manifest[ontology_hash]` → re-sync ontology
- Otherwise: skip — no Athena read, no Bodhi call, no cost

In [ ]:
MANIFEST_KEY = f'{SYNC_PREFIX}/manifest.json'

def load_manifest() -> dict:
    """Load manifest from S3, or return empty manifest if it doesn't exist yet."""
    try:
        obj = s3.get_object(Bucket=SYNC_BUCKET, Key=MANIFEST_KEY)
        manifest = json.loads(obj['Body'].read())
        print('  → Existing manifest loaded from S3')
        return manifest
    except s3.exceptions.NoSuchKey:
        print('  → No manifest found — this is the first sync run')
        return {'snapshots': {}, 'ontology_hash': None, 'last_sync': None}

print('Loading manifest ...')
manifest = load_manifest()
print(json.dumps(manifest, indent=2, default=str))

In [ ]:
# ── Schema diff ───────────────────────────────────────────────────────────────
print('=== Schema change detection ===')
current_snapshot_ids = {
    t: (snapshots[t].get('snapshot_id') if snapshots.get(t) else None)
    for t in ALLOWED_TABLES
}

tables_to_sync = []
for table, snap_id in current_snapshot_ids.items():
    prev = manifest['snapshots'].get(table)
    changed = (prev != snap_id)
    status  = 'CHANGED  ← will sync' if changed else 'unchanged'
    print(f'  {table:<15} prev={str(prev)[:16]:<18} now={str(snap_id)[:16]:<18} {status}')
    if changed:
        tables_to_sync.append(table)

print()

# ── Ontology diff ─────────────────────────────────────────────────────────────
print('=== Ontology change detection ===')
onto_prev = manifest.get('ontology_hash')
onto_changed = (onto_prev != content_hash)
print(f'  prev hash : {onto_prev}')
print(f'  curr hash : {content_hash}')
print(f'  status    : {"CHANGED  ← will sync" if onto_changed else "unchanged"}')

---
## Part D — Upload to S3 (Bodhi Placeholder)

Since the Bodhi API is not yet live, we upload the JSON payloads to S3 as a
placeholder. The S3 objects are API-ready — when Bodhi is available, the upload
step is simply replaced with an HTTP POST.

We write **two copies** of each payload:
- A **timestamped version** (`schema_20261107_143022.json`) for history and audit
- A **`_latest` file** (`schema_latest.json`) for Bodhi to always read the most recent

After uploading, we write an updated **manifest** so the next run knows what to skip.

In [ ]:
def upload_json(key: str, payload: dict) -> str:
    """Upload a dict as JSON to S3 and return the full S3 URI."""
    body = json.dumps(payload, indent=2, default=str).encode()
    s3.put_object(
        Bucket=SYNC_BUCKET,
        Key=key,
        Body=body,
        ContentType='application/json'
    )
    uri = f's3://{SYNC_BUCKET}/{key}'
    print(f'  ✓ Uploaded → {uri}  ({len(body):,} bytes)')
    return uri


ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
uploads = []

# Upload schema payload (if any table changed)
if tables_to_sync:
    print('Uploading schema payload ...')
    uploads.append(upload_json(f'{SYNC_PREFIX}/metadata/schema_{ts}.json',    schema_payload))
    uploads.append(upload_json(f'{SYNC_PREFIX}/metadata/schema_latest.json',  schema_payload))
else:
    print('  Schema unchanged — skipping upload')

# Upload ontology payload (if content changed)
if onto_changed:
    print('\nUploading ontology payload ...')
    uploads.append(upload_json(f'{SYNC_PREFIX}/ontology/ontology_{ts}.json',   ontology_payload))
    uploads.append(upload_json(f'{SYNC_PREFIX}/ontology/ontology_latest.json', ontology_payload))
else:
    print('\n  Ontology unchanged — skipping upload')

print(f'\n  {len(uploads)} objects uploaded')

In [ ]:
# Update manifest with current state
manifest['snapshots'].update(current_snapshot_ids)
manifest['ontology_hash'] = content_hash
manifest['last_sync']     = synced_at

print('Saving updated manifest ...')
s3.put_object(
    Bucket=SYNC_BUCKET,
    Key=MANIFEST_KEY,
    Body=json.dumps(manifest, indent=2, default=str).encode(),
    ContentType='application/json'
)
print(f'  ✓ Manifest saved → s3://{SYNC_BUCKET}/{MANIFEST_KEY}')
print()
print('Final manifest:')
print(json.dumps(manifest, indent=2, default=str))

---
## Summary — Full Sync Result

In [ ]:
print('=' * 60)
print('  BODHI SYNC — COMPLETE')
print('=' * 60)
print(f'  sync_id          : {sync_id}')
print(f'  synced_at        : {synced_at}')
print(f'  schema synced    : {bool(tables_to_sync)}  {tables_to_sync if tables_to_sync else "(no changes)"}')
print(f'  ontology synced  : {onto_changed}')
print(f'  files uploaded   : {len(uploads)}')
print()
print('S3 layout:')
print(f'  s3://{SYNC_BUCKET}/{SYNC_PREFIX}/')
print(f'    manifest.json')
print(f'    metadata/')
print(f'      schema_latest.json          ← always current')
print(f'      schema_{ts}.json  ← versioned copy')
print(f'    ontology/')
print(f'      ontology_latest.json        ← always current')
print(f'      ontology_{ts}.json← versioned copy')
print()
print('Next step: replace S3 upload with HTTP POST to Bodhi API endpoint.')

---
## What happens when Bodhi API is ready?

Replace the `upload_json()` calls above with:

```python
import httpx

BODHI_API = 'https://bodhi.internal/api/v1/ingest'

if tables_to_sync:
    resp = httpx.post(f'{BODHI_API}/schema',   json=schema_payload,   timeout=30)
    resp.raise_for_status()

if onto_changed:
    resp = httpx.post(f'{BODHI_API}/ontology', json=ontology_payload, timeout=30)
    resp.raise_for_status()
```

The payload format is identical — no changes to the extraction or manifest logic.

---
*YumBurger Analytics · Bodhi Sync Prototype*